# 04. Pandas 데이터 결합 - 실습 문제

## 실습 안내
- 총 10개 문제
- concat, merge를 활용한 다중 테이블 결합
- 실제 제조 데이터 통합 분석 시나리오
- 생산/품질/설비/정비 데이터 종합 활용

## 데이터 로드

In [1]:
import pandas as pd
import numpy as np

# 데이터 불러오기
production_df = pd.read_csv('../data/05_production.csv', encoding='utf-8-sig')
quality_df = pd.read_csv('../data/07_quality_inspection.csv', encoding='utf-8-sig', na_values=['\\N'])
equipment_df = pd.read_csv('../data/01_equipment.csv', encoding='utf-8-sig')
sensor_df = pd.read_csv('../data/08_sensor_data.csv', encoding='utf-8-sig')
maintenance_df = pd.read_csv('../data/10_maintenance_history.csv', encoding='utf-8-sig')
operation_df = pd.read_csv('../data/06_equipment_operation.csv', encoding='utf-8-sig')
product_df = pd.read_csv('../data/02_product.csv', encoding='utf-8-sig')

# 기본 전처리
production_df['production_date'] = pd.to_datetime(production_df['production_date'])
quality_df['inspection_time'] = pd.to_datetime(quality_df['inspection_time'])

print("데이터 로드 완료!")
print(f"생산: {len(production_df):,}건")
print(f"품질: {len(quality_df):,}건")
print(f"설비: {len(equipment_df):,}건")
print(f"센서: {len(sensor_df):,}건")
print(f"정비: {len(maintenance_df):,}건")
print(f"설비운영: {len(operation_df):,}건")
print(f"제품: {len(product_df):,}건")

데이터 로드 완료!
생산: 1,872건
품질: 37,417건
설비: 5건
센서: 10,920건
정비: 98건
설비운영: 3,304건
제품: 3건


---
## 문제 1: 생산 데이터에 설비 정보 추가

**시나리오**: 생산 데이터에 설비의 상세 정보를 결합하여 분석을 용이하게 하세요.

**요구사항**:
1. production_df와 equipment_df를 equipment_id로 결합 (left join)
2. 필요한 컬럼만 선택:
   - equipment_df에서: equipment_name, equipment_type, manufacturer, capacity
3. 결합된 데이터의 처음 10개 행 출력
4. 결합 전후 데이터 건수 확인

**힌트**: `pd.merge()`, how='left', on='equipment_id'

In [82]:
equipment_df.columns

Index(['equipment_id', 'equipment_name', 'equipment_type', 'location',
       'rated_capacity', 'installation_date', 'status', 'created_at',
       'updated_at'],
      dtype='object')

In [85]:
# 여기에 코드 작성
equipment_sum = equipment_df[['equipment_id', 'equipment_name', 'equipment_type', 'rated_capacity']]
production_eqp = pd.merge(production_df, equipment_sum, how = 'left', on = 'equipment_id')

In [13]:
print(production_df.shape)
print(production_eqp.shape)

(1872, 17)
(1872, 20)


---
## 문제 2: 생산 데이터에 제품 정보 추가

**시나리오**: 생산 데이터에 제품 상세 정보를 결합하세요.

**요구사항**:
1. production_df와 product_df를 product_code로 결합 (left join)
2. 제품명(product_name) 추가
3. 제품명별 생산 건수 집계
4. 제품명별 평균 불량률 계산 (defect_quantity / actual_quantity)

**힌트**: merge 후 groupby로 집계

In [17]:
product_df.columns

Index(['product_code', 'product_name', 'specification', 'unit',
       'standard_cycle_time', 'target_quality_rate', 'upper_spec_limit',
       'lower_spec_limit', 'created_at', 'updated_at'],
      dtype='object')

In [87]:
# 여기에 코드 작성
prodct_temp = product_df[['product_code', 'product_name']]
production_prd = pd.merge(production_df, prodct_temp, how = 'left', on = 'product_code')
production_prd.head()

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,product_name
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,범퍼
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,범퍼
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,범퍼
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,대시보드
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,도어패널


In [90]:
production_prd['product_name'].value_counts()
production_prd.groupby('product_name')['product_name'].count()

product_name
대시보드    583
도어패널    641
범퍼      648
Name: product_name, dtype: int64

In [92]:
production_prd['불량률'] = (production_prd['defect_quantity'] / production_prd['actual_quantity'] * 100).round(2)
production_prd.groupby('product_name')['불량률'].mean()

product_name
대시보드    10.121681
도어패널    10.158986
범퍼      10.376590
Name: 불량률, dtype: float64

---
## 문제 3: 생산 데이터와 품질 검사 결합

**시나리오**: 각 생산 건에 대한 품질 검사 정보를 결합하세요.

**요구사항**:
1. production_df와 quality_df를 production_id로 결합 (left join)
2. 결합 후 데이터 건수 확인 (1:N 관계 주의)
3. production_id별 검사 건수 계산
4. 검사가 없는 생산 건 찾기

**힌트**: left join, 결합 후 value_counts(), isna() 확인

In [24]:
# 여기에 코드 작성
pd_quality = pd.merge(production_df, quality_df, how = 'left', on = 'production_id')
print(production_df.shape)
print(quality_df.shape)
print(pd_quality.shape)

(1872, 17)
(37417, 15)
(37417, 31)


In [93]:
pd_quality.groupby('production_id')['production_id'].count().isna()

production_id
1       False
2       False
3       False
4       False
5       False
        ...  
1868    False
1869    False
1870    False
1871    False
1872    False
Name: production_id, Length: 1872, dtype: bool

In [98]:
production_df[~production_df['production_id'].isin(quality_df['production_id'])]
(~production_df['production_id'].isin(quality_df['production_id'])).sum()

np.int64(0)

---
## 문제 4: 품질 검사 데이터 집계 후 생산 데이터와 결합

**시나리오**: production_id별 품질 검사 결과를 집계한 후 생산 데이터에 추가하세요.

**요구사항**:
1. quality_df를 production_id로 그룹화하여 집계:
   - 검사 건수
   - 불량 건수 (result='FAIL')
   - 평균 측정값
2. 집계 결과를 production_df와 결합 (left join)
3. 검사가 없는 경우 0으로 채우기
4. 처음 10개 행 출력

**힌트**: groupby + agg, reset_index(), merge, fillna(0)

In [43]:
quality_df.columns

Index(['inspection_id', 'production_id', 'equipment_id', 'product_code',
       'inspection_time', 'inspection_type', 'result', 'defect_code',
       'measurement_value', 'measurement_unit', 'inspector_id', 'lot_no',
       'sample_size', 'notes', 'created_at'],
      dtype='object')

In [99]:
#def fail_len(x):
#    return (x == 'FAIL').sum()

#quality_gb = quality_df.groupby('production_id').agg({'inspection_id':'count',
#                                         'result' : fail_len,
#                                         'measurement_value' : 'mean'}).fillna(0)

In [51]:
# 여기에 코드 작성
quality_gb = quality_df.groupby('production_id').agg({'inspection_id':'count',
                                         'result' : lambda x: (x == 'FAIL').sum(),
                                         'measurement_value' : 'mean'}).fillna(0)

quality_gb.columns = ['검사건수', '불량건수', '평균측정값']

In [53]:
pd.merge(production_df, quality_gb, how = 'left', on = 'production_id').head(10)

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,검사건수,불량건수,평균측정값
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,11,4,300.583600
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,13,6,299.203377
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,13,3,298.295115
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,11,2,400.704636
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17,7,598.715941
5,6,INJ-002,BUMPER-A,2024-01-01,2024-01-02 00:22:00,2024-01-02 03:10:05,124,130,123,7,77.58,WO202401015333,LOT2024010100211,OP005,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17,7,301.318788
6,7,PRESS-001,DASH-C,2024-01-01,2024-01-01 10:07:00,2024-01-01 12:48:06,128,106,102,4,91.20,WO202401015803,LOT2024010100101,OP010,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,14,4,402.000014
7,8,PRESS-001,DASH-C,2024-01-01,2024-01-01 14:12:00,2024-01-01 15:58:01,88,73,70,3,87.14,WO202401014733,LOT2024010100102,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,10,3,402.013350
8,9,PRESS-001,DOOR-B,2024-01-01,2024-01-01 20:55:00,2024-01-01 22:27:01,92,90,84,6,61.35,WO202401017227,LOT2024010100110,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,14,6,601.626300
9,10,PRESS-001,DOOR-B,2024-01-01,2024-01-02 02:53:00,2024-01-02 05:09:08,126,123,115,8,66.41,WO202401013664,LOT2024010100111,OP005,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,18,8,601.272567


---
## 문제 5: 설비 + 생산 + 정비 종합 분석

**시나리오**: 설비별 생산 실적과 정비 이력을 종합하여 설비 성능을 평가하세요.

**요구사항**:
1. production_df를 설비별로 집계:
   - 생산 건수, 총 생산량
2. maintenance_df를 설비별로 집계:
   - 정비 건수, 총 정비 비용, 총 정지 시간
3. equipment_df에 위 두 집계 결과를 순차적으로 결합 (left join)
4. 결측치는 0으로 채우기
5. 설비당 평균 정비 비용 계산

**힌트**: 각각 집계 후 reset_index(), 순차적 merge

In [54]:
production_df.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at'],
      dtype='object')

In [55]:
maintenance_df.columns

Index(['equipment_id', 'maintenance_type', 'start_time', 'end_time',
       'maintenance_description', 'parts_replaced', 'cost', 'technician_id',
       'downtime_hours', 'notes', 'created_at'],
      dtype='object')

In [59]:
# 여기에 코드 작성
prod_sum = production_df.groupby('equipment_id').agg({'production_id':'count',
                                           'actual_quantity' : 'sum'})
prod_sum.columns = ['생산건수', '총생산량']
main_sum = maintenance_df.groupby('equipment_id').agg({'maintenance_type':'count',
                                            'cost' : 'sum',
                                            'downtime_hours' : 'sum'})
main_sum.columns = ['정비건수', '총정비비용', '총정지시간']
equipment_analysis = pd.merge(equipment_df, prod_sum, how = 'left', on = 'equipment_id').fillna(0)
equipment_analysis = pd.merge(equipment_analysis, main_sum, how = 'left', on = 'equipment_id').fillna(0)

In [73]:
equipment_analysis['설비당평균정비비용'] = (equipment_analysis['총정비비용'] / equipment_analysis['정비건수']).round(2)

In [74]:
equipment_analysis

,equipment_id,equipment_name,equipment_type,location,rated_capacity,installation_date,status,created_at,updated_at,생산건수,총생산량,정비건수,총정비비용,총정지시간,설비당평균정비비용
0,INJ-001,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,262,28163,17,45291758.85,27.85,2664221.11
1,INJ-002,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,430,51958,23,75076538.50,40.36,3264197.33
2,PRESS-001,프레스 1호기,프레스,A동 2라인,200.0,2019-05-10,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,468,52069,18,44564270.41,29.69,2475792.80
3,PRESS-002,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,478,51929,21,70921381.60,37.71,3377208.65
4,ASM-001,조립라인 1호기,조립라인,B동 1라인,100.0,2020-11-30,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,234,22485,19,60072924.01,30.79,3161732.84


---
## 문제 6: 생산량과 설비 capacity 비교

**시나리오**: 설비의 capacity 대비 실제 생산량을 비교하여 가동률을 분석하세요.

**요구사항**:
1. production_df와 equipment_df를 결합 (설비 정보 추가)
2. 각 생산 건의 capacity 대비 생산량 비율 계산:
   - capacity_utilization = (actual_quantity / capacity) * 100
3. 설비별 평균 capacity_utilization 계산
4. 가동률이 가장 높은 상위 5개 설비 출력

**힌트**: merge, 새 컬럼 생성, groupby, sort_values()

In [78]:
# 여기에 코드 작성
prod_equip = pd.merge(production_df, equipment_df, how = 'left', on = 'equipment_id')
prod_equip['capacity_utilization'] = prod_equip['actual_quantity'] / prod_equip['rated_capacity'] * 100
prod_equip.groupby('equipment_id').agg({'capacity_utilization':'mean'}).sort_values(by = 'capacity_utilization', ascending =False).reset_index()

,equipment_id,capacity_utilization
0,ASM-001,96.089744
1,INJ-002,80.555039
2,INJ-001,71.661578
3,PRESS-001,55.629274
4,PRESS-002,54.319038


---
## 문제 7: 제조사별 설비 성능 분석

**시나리오**: 설비 제조사별로 생산 성능을 비교 분석하세요.

**요구사항**:
1. production_df에 equipment_df를 결합 (manufacturer 정보 추가)
2. 제조사별로 다음 집계:
   - 설비 수 (equipment_id의 unique count)
   - 총 생산량
   - 평균 불량률
   - 평균 사이클 타임
3. 총 생산량 기준 내림차순 정렬

**힌트**: merge, groupby with nunique, agg

In [80]:
equipment_df.head()

,equipment_id,equipment_name,equipment_type,location,rated_capacity,installation_date,status,created_at,updated_at
0,INJ-001,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1,INJ-002,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
2,PRESS-001,프레스 1호기,프레스,A동 2라인,200.0,2019-05-10,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
3,PRESS-002,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
4,ASM-001,조립라인 1호기,조립라인,B동 1라인,100.0,2020-11-30,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00


In [79]:
# 여기에 코드 작성
prod_equip = pd.merge(production_df, equipment_df, how='left', on='equipment_id')
prod_equip.head(2)

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,created_at_x,updated_at_x,equipment_name,equipment_type,location,rated_capacity,installation_date,status,created_at_y,updated_at_y
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00


---
## 문제 8: 월별 데이터 분할 후 재결합

**시나리오**: 생산 데이터를 월별로 분할하여 각각 분석한 후 다시 합치세요.

**요구사항**:
1. production_df를 1월, 2월, 3월 데이터로 분할
2. 각 월의 데이터에 '분기' 컬럼 추가 (모두 'Q1')
3. concat으로 세 DataFrame을 세로로 결합
4. 결합 후 인덱스 재설정 (ignore_index=True)
5. 분기별 생산 건수 확인

**힌트**: 월별 필터링, concat([df1, df2, df3], axis=0)

In [103]:
# 여기에 코드 작성
df_1 = production_df.loc[ production_df['production_date'].dt.month == 1 , ].copy()
df_2 = production_df.loc[ production_df['production_date'].dt.month == 2 , ].copy()
df_3 = production_df.loc[ production_df['production_date'].dt.month == 3 , ].copy()
df_1['분기'] = 'Q1'
df_2['분기'] = 'Q1'
df_3['분기'] = 'Q1'

In [104]:
prod_q1 = pd.concat([df_1, df_2, df_3], ignore_index = True)

In [106]:
prod_q1['분기'].value_counts()

분기
Q1    1872
Name: count, dtype: int64

---
## 문제 9: 생산-품질-설비 3중 결합

**시나리오**: 생산, 품질, 설비 데이터를 모두 결합하여 종합 분석 테이블을 만드세요.

**요구사항**:
1. 품질 데이터를 production_id별로 집계:
   - 검사 건수, 불량 건수, 평균 측정값
2. 생산 데이터에 품질 집계 결합 (left join)
3. 결과에 설비 정보 결합 (equipment_name, equipment_type 추가)
4. 설비 타입별 평균 불량률 계산
5. 결과를 설비 타입으로 그룹화하여 출력

**힌트**: 순차적 merge, 중간 결과 확인하며 진행

In [107]:
# 여기에 코드 작성
qu_agg = quality_df.groupby('production_id').agg({'inspection_id' : 'count',
                                         'result' : lambda x: (x=='FAIL').sum(),
                                         'measurement_value' : 'mean'})
qu_agg.columns = ['검사건수', '불량건수', '평균측정값']

In [109]:
eq_df = equipment_df[['equipment_id', 'equipment_name', 'equipment_type']]

In [111]:
prod_all = pd.merge(production_df, qu_agg, on = 'production_id', how='left')
prod_all = pd.merge(prod_all, eq_df, on = 'equipment_id', how = 'left')

In [112]:
prod_all.head()

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,lot_no,operator_id,shift,created_at,updated_at,검사건수,불량건수,평균측정값,equipment_name,equipment_type
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,...,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,11,4,300.583600,사출기 1호기,사출기
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,...,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,13,6,299.203377,사출기 1호기,사출기
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,...,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,13,3,298.295115,사출기 2호기,사출기
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,...,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,11,2,400.704636,사출기 2호기,사출기
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,...,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17,7,598.715941,사출기 2호기,사출기


---
## 문제 10: 종합 설비 대시보드 데이터 생성

**시나리오**: 모든 데이터를 결합하여 설비별 종합 성능 대시보드를 위한 데이터를 생성하세요.

**요구사항**:
1. 설비별 생산 집계:
   - 생산 건수, 총 생산량, 평균 불량률
2. 설비별 정비 집계:
   - 정비 건수, 총 정비 비용, 평균 정지 시간
3. 설비별 설비운영 집계:
   - RUNNING 시간 합계 (operation_status='RUNNING'인 경우)
4. equipment_df를 기준으로 위 3개 집계를 모두 결합 (left join)
5. 결측치를 0으로 채우기
6. 다음 지표 계산:
   - 가동률 = (RUNNING 시간 / 전체 시간) * 100
   - 생산성 = 총 생산량 / 생산 건수
   - 정비 효율 = 총 생산량 / 정비 건수
7. 종합 점수 = 생산성 * (100 - 평균불량률) / 정비건수
8. 종합 점수 상위 5개 설비 출력

**힌트**: 복잡한 문제. 단계별로 진행, 각 집계 후 reset_index(), 순차적 merge, 계산 컬럼 생성

In [ ]:
# 여기에 코드 작성


---
## 수고하셨습니다!

### 학습 체크리스트
- [ ] concat으로 DataFrame 이어붙이기
- [ ] merge의 다양한 join 타입 이해 (inner, left, right, outer)
- [ ] 1:1, 1:N 관계 이해
- [ ] 집계 후 결합하기
- [ ] 순차적 다중 테이블 결합
- [ ] 결합 후 결측치 처리
- [ ] 복합 분석을 위한 데이터 통합
- [ ] 실무 분석 시나리오 적용

